# Register `next_best_action_agent` to Unity Catalog

This notebook creates a minimal LangChain runnable, logs it with MLflow, and registers it in Unity Catalog.

Target model name:
- `<catalog>.<schema>.next_best_action_agent`

In [ ]:
%pip install -qU mlflow "langchain<1.0.0"

In [ ]:
dbutils.library.restartPython()

In [ ]:
import mlflow
from mlflow.models import infer_signature
from langchain_core.runnables import RunnableLambda

# Optional job parameters
dbutils.widgets.text("catalog", "bx4")
dbutils.widgets.text("schema", "butterfly")

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
model_name = f"{catalog}.{schema}.next_best_action_agent"

mlflow.set_tracking_uri("databricks")
mlflow.set_registry_uri("databricks-uc")

chain = RunnableLambda(
    lambda payload: (
        "Recommended action: Share role-specific follow-up resources and schedule a 15-minute check-in. "
        f"Context used: {payload.get('context', '')[:300]}"
    )
)

input_example = {
    "context": "Cardiovascular surgeon at Valley Regional Hospital with high intent score."
}

example_output = chain.invoke(input_example)
signature = infer_signature(input_example, example_output)

with mlflow.start_run(run_name="register_next_best_action_agent_demo"):
    model_info = mlflow.langchain.log_model(
        lc_model=chain,
        artifact_path="langchain-model",
        input_example=input_example,
        signature=signature,
    )
    registered = mlflow.register_model(model_info.model_uri, model_name)

print(f"Registered model: {registered.name} v{registered.version}")
print(f"Model URI: models:/{registered.name}/{registered.version}")